# Milestone 3A — Verify the shared PyTorch data loaders

**Objective:** load the existing frozen Pakistani and Stanford Cars manifests with one reusable PyTorch `Dataset`, without reshuffling or modifying any images or split assignments.

- **Pakistan:** 2,610 train, 463 val, 751 test; 48 classes.
- **Stanford Cars:** 6,893 train, 1,214 val, 8,041 test; 196 classes (37 excluded after overlap review).
- Both Xception and ConvNeXt-Tiny will use these exact assignments. The transforms in this notebook are **only for a quick data-loading smoke test**. We will configure training augmentation and model-specific pretrained normalization in Milestone 3B before training.

Run from VS Code using your existing project environment. No GPU is required.

## 1. Find the repository root and import our reusable loader

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for folder in (start, *start.parents):
        if (folder / "configs/experiments.yaml").is_file() and (folder / "splits/pakistan/final_manifest.csv").is_file():
            return folder
    raise FileNotFoundError("Repository root not found. Open notebook from within the repository.")

ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))

from vehicle_data import VehicleManifestDataset, inspect_manifest
print("Repo:", ROOT)

## 2. Re-check the frozen manifests and all image paths

`inspect_manifest()` validates the 0-based class mappings, checks all three splits contain all expected classes, and checks every active image path exists. For Stanford, the module also compares the manifest SHA-256 to the frozen verification summary. It **never** creates new splits.

In [ ]:
expected = {
    "pakistan": {"train": 2610, "val": 463, "test": 751, "excluded_test_overlap": 23},
    "stanford_cars": {"train": 6893, "val": 1214, "test": 8041, "excluded": 37},
}

for dataset_name in expected:
    info = inspect_manifest(ROOT, dataset_name, check_files=True)
    print(f"\n{dataset_name}:")
    print("  Class count:", info["class_count"])
    print("  Split counts:", info["counts"])
    print("  Manifest SHA-256:", info["manifest_sha256"])
    for split_name, n in expected[dataset_name].items():
        assert info["counts"].get(split_name, 0) == n, (dataset_name, split_name, info["counts"])
print("\nPASS: both frozen manifests match the expected counts and all active files exist.")

## 3. Simple *evaluation-style* transform for a smoke test

All images are converted to RGB by `VehicleManifestDataset`. `Resize(256)` scales the shorter side to 256 pixels; `CenterCrop(224)` extracts a consistent central 224×224 square; `ToTensor()` maps pixel values into `[0,1]` and rearranges shape to `[channels, height, width]`.

**Important:** This is **not** the final pretrained model input. In Milestone 3B we will add the correct normalization for each pretrained checkpoint and implement the fixed shared training augmentation specified in `configs/experiments.yaml`.

In [ ]:
import torch
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torch.utils.data import DataLoader

smoke_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
])

print("Torch:", torch.__version__)
print("Smoke transform:", smoke_transform)

## 4. Load a small batch from every split of both datasets

We set `num_workers=0` for easier first-run debugging on your Mac. When training in Colab, we will use the configured worker count (currently 2) and set the DataLoader random seed. The exact image assignments stay unchanged regardless of batch size or shuffling.

In [ ]:
datasets_by_name = {}
for dataset_name in ("pakistan", "stanford_cars"):
    datasets_by_name[dataset_name] = {}
    for split_name in ("train", "val", "test"):
        dataset = VehicleManifestDataset(ROOT, dataset_name, split_name, transform=smoke_transform)
        datasets_by_name[dataset_name][split_name] = dataset
        loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=0)
        images, labels = next(iter(loader))
        assert images.shape == (4, 3, 224, 224)
        assert labels.dtype == torch.int64
        print(f"{dataset_name:13} {split_name:5} n={len(dataset):5} images={tuple(images.shape)} labels={labels.tolist()}")
print("\nPASS: both datasets produce valid PyTorch image tensors and integer labels.")

## 5. Show one batch so we can inspect the inputs visually

In [ ]:
import matplotlib.pyplot as plt

chosen_name = "stanford_cars"   # Change to "pakistan" to inspect the other dataset.
chosen = datasets_by_name[chosen_name]["train"]
images, labels = next(iter(DataLoader(chosen, batch_size=4, shuffle=False, num_workers=0)))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, image, label in zip(axes, images, labels):
    ax.imshow(image.permute(1, 2, 0).numpy())
    ax.set_title(chosen.classes[int(label)], fontsize=8, wrap=True)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Completion criteria

Milestone 3A passes when both manifest audits report the expected counts, every active image exists, both data loaders output `(batch, 3, 224, 224)` tensors, and the displayed labels match the pictured vehicles in the small visual check.

**Next:** Milestone 3B — instantiate `legacy_xception.tf_in1k` and `convnext_tiny.fb_in1k` via `timm`, read their pretrained normalization settings, and implement the common training/validation transformations and two-stage transfer-learning procedure. No accuracy or timing measurements should be entered in the experiment log until actual training runs are completed.